# Intelligent Healthcare Assistant – Notebook Overview

This notebook demonstrates the complete pipeline of an intelligent healthcare assistant that processes user symptoms, predicts possible diseases, and suggests supportive home remedies.

## Part 1 – Symptom Extraction from User Conversation

In this section, user input provided in natural language is processed using basic NLP techniques. The system identifies relevant symptoms from free-text conversation, normalizes them, and prepares them in a structured format. This step ensures that unstructured user descriptions can be converted into machine-readable symptom features.

## Part 2 – Disease Prediction using Pre-trained XGBoost Model

Here, the extracted symptoms are transformed into a binary feature vector that matches the format used during model training. The pre-trained XGBoost classifier then predicts the most probable disease along with confidence probabilities. This step represents the core machine learning component of the system.

## Part 3 – Home Remedy Recommendation

In the final section, the predicted disease is used to retrieve suitable home remedies from a curated knowledge base. The system combines multiple remedies if available and generates a user-friendly response along with a safety disclaimer. This provides supportive guidance while encouraging professional medical consultation when required.

# Part 1 - Symptom Extraction from User Conversation

In [12]:
!pip install spacy rapidfuzz
!python -m spacy download en_core_web_sm
import pandas as pd

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 6.1 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.8 MB 9.5 MB/s eta 0:00:01
     --------------------- ------------------ 6.8/12.8 MB 11.3 MB/s eta 0:00:01
     ------------------------------ -------- 10.0/12.8 MB 12.4 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 12.7 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 12.4 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [14]:
#Read the data set of symptoms and diseases
df = pd.read_csv("C://Users//kalya//Desktop//Kalyan//Upgrad AIML//Thesis Topics//dataset.csv")

## Pre-Process the symptoms data

In [17]:
# List of symptom columns
symptom_columns = [f'Symptom_{i}' for i in range(1, 18)]  # Symptom_1 to Symptom_17

# Flatten and get unique symptoms
SYMPTOM_VOCAB = pd.unique(df[symptom_columns].values.ravel())
SYMPTOM_VOCAB = [symptom for symptom in SYMPTOM_VOCAB if pd.notnull(symptom)]  # Remove NaN
SYMPTOM_VOCAB = [s.strip() for s in SYMPTOM_VOCAB]
print (SYMPTOM_VOCAB)

['itching', 'skin_rash', 'nodal_skin_eruptions', 'dischromic _patches', 'continuous_sneezing', 'shivering', 'chills', 'watering_from_eyes', 'stomach_pain', 'acidity', 'ulcers_on_tongue', 'vomiting', 'cough', 'chest_pain', 'yellowish_skin', 'nausea', 'loss_of_appetite', 'abdominal_pain', 'yellowing_of_eyes', 'burning_micturition', 'spotting_ urination', 'passage_of_gases', 'internal_itching', 'indigestion', 'muscle_wasting', 'patches_in_throat', 'high_fever', 'extra_marital_contacts', 'fatigue', 'weight_loss', 'restlessness', 'lethargy', 'irregular_sugar_level', 'blurred_and_distorted_vision', 'obesity', 'excessive_hunger', 'increased_appetite', 'polyuria', 'sunken_eyes', 'dehydration', 'diarrhoea', 'breathlessness', 'family_history', 'mucoid_sputum', 'headache', 'dizziness', 'loss_of_balance', 'lack_of_concentration', 'stiff_neck', 'depression', 'irritability', 'visual_disturbances', 'back_pain', 'weakness_in_limbs', 'neck_pain', 'weakness_of_one_body_side', 'altered_sensorium', 'dark_

In [19]:

# Stop phrases to indicate that the conversation can end. This will be honoured after a minimum of 3 symptoms are collected

STOP_PHRASES = {
    "stop", "no", "no more", "that's all", "thats all",
    "done", "enough", "nothing else", "quit"
}

import re

def normalize_user_text(text: str) -> str:
    """
    Normalize user input:
    - lowercase
    - remove punctuation
    - normalize spaces
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_vocab_symptom(symptom: str) -> str:
    """
    Convert vocab symptom to comparison-friendly format:
    - high_fever -> high fever
    """
    return symptom.lower().replace("_", " ")


# -----------------------------------------------------
# FUNCTIONS that are needed to process the conversation
# -----------------------------------------------------

def user_wants_to_stop(text: str) -> bool:
    text = text.lower()
    return any(p in text for p in STOP_PHRASES)


def normalize_text(text: str) -> str:
    return text.lower().strip()


# This funciton extracts the symptoms from the user conversation
def extract_symptoms_from_text(user_text: str, vocab):
    """
    Extract symptoms from free-text user input.
    - Matches underscore-based vocab (high_fever)
    - Against space-based user input (high fever)
    - Case-insensitive
    """
    text = normalize_user_text(user_text)
    found = []

    for symptom in vocab:
        symptom_cmp = normalize_vocab_symptom(symptom)

        # Exact phrase match with word boundaries
        pattern = r"\b" + re.escape(symptom_cmp) + r"\b"

        if re.search(pattern, text):
            found.append({
                "symptom": symptom,   # keep underscore version for ML
                "negated": False
            })

    return found



# ------------------------------
# MAIN CONVERSATION FUNCTION
# ------------------------------

def converse_collect_symptoms_min3_max8(user_input_fn, assistant_say_fn, vocab=SYMPTOM_VOCAB):
    """
    Collects between 3 and 8 positive symptoms.
    Stops early if user asks to stop after minimum is met.
    """
    collected = []
    asked_after_min = False

    assistant_say_fn(
        "Hi — please describe your symptoms in your own words. "
        "You can mention up to 8 symptoms. Type 'stop' when you're done."
    )

    while True:
        user_text = user_input_fn()

        if not user_text or user_text.strip() == "":
            assistant_say_fn(
                "I didn't catch that — could you describe any symptoms "
                "(for example: fever, cough, headache)?"
            )
            continue

        # Stop intent
        if user_wants_to_stop(user_text):
            if len(collected) >= 3:
                assistant_say_fn("Okay — stopping symptom collection.")
                break
            else:
                assistant_say_fn(
                    f"I need at least {3 - len(collected)} more symptom(s) "
                    "before I can proceed."
                )
                continue

        extracted = extract_symptoms_from_text(user_text, vocab=vocab)

        added_any = False
        for e in extracted:
            if not e["negated"] and e["symptom"] not in collected:
                collected.append(e["symptom"])
                assistant_say_fn(f"Noted: **{e['symptom']}**.")
                added_any = True

                if len(collected) >= 8:
                    assistant_say_fn("You've reached the maximum of 8 symptoms.")
                    return collected[:8]

        if not added_any:
            assistant_say_fn(
                "I couldn't clearly identify a new symptom there. "
                "Could you describe it differently or mention another symptom?"
            )

        if len(collected) >= 3 and not asked_after_min:
            assistant_say_fn(
                "I have enough information to proceed. "
                "Would you like to add more symptoms? "
                "You can add up to 8 or type 'stop'."
            )
            asked_after_min = True

    assistant_say_fn(
        f"Thanks — I have recorded these symptoms: {', '.join(collected)}."
    )
    return collected


# ----------------------------------------------------------------------------
# Main function, that calls the symptom collection funcitons and process them
# ----------------------------------------------------------------------------

if __name__ == "__main__":

    def user_input_fn():
        return input("You: ")

    def assistant_say_fn(text):
        print("Assistant:", text)

    collected_symptoms = converse_collect_symptoms_min3_max8(
        user_input_fn=user_input_fn,
        assistant_say_fn=assistant_say_fn,
        vocab=SYMPTOM_VOCAB
    )

    print("\n✅ Final collected symptoms:")
    for i, s in enumerate(collected_symptoms, start=1):
        print(f"{i}. {s}")


Assistant: Hi — please describe your symptoms in your own words. You can mention up to 8 symptoms. Type 'stop' when you're done.


You:  i am having cough since 2 days


Assistant: Noted: **cough**.


You:  i see fatigue past one day


Assistant: Noted: **fatigue**.


...

Assistant: Noted: **breathlessness**.


You:  and this is in my family history


Assistant: Noted: **family_history**.


You:  stop


Assistant: Okay — stopping symptom collection.
Assistant: Thanks — I have recorded these symptoms: cough, fatigue, high_fever, breathlessness, family_history.

✅ Final collected symptoms:
1. cough
2. fatigue
3. high_fever
4. breathlessness
5. family_history


# Part 2 - Predict the disease using a pre-trained XGBoost model

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [23]:
import numpy as np
import joblib

# Load model artifacts
xgb_model = joblib.load("xgboost_final_full_data_model.joblib")
CLASS_NAMES = joblib.load("xgboost_class_names.joblib")
FEATURE_COLUMNS = joblib.load("xgboost_feature_columns.joblib")

def normalize_symptom(symptom):
    return symptom.strip().lower().replace(" ", "_")

symptoms = [normalize_symptom(s) for s in collected_symptoms]

def symptoms_to_binary_vector(symptoms, feature_columns):
    vector = np.zeros(len(feature_columns), dtype=int)
    for i, col in enumerate(feature_columns):
        if col in symptoms:
            vector[i] = 1
    return vector.reshape(1, -1)

X_input = symptoms_to_binary_vector(symptoms, FEATURE_COLUMNS)

# Predict
predicted_class = int(xgb_model.predict(X_input)[0])
predicted_proba = xgb_model.predict_proba(X_input)[0]

predicted_disease = CLASS_NAMES[predicted_class]

print("\nPredicted Disease:", predicted_disease)

# Top-3 predictions
top3_idx = np.argsort(predicted_proba)[-3:][::-1]

print("\nTop-3 Predictions:")
for idx in top3_idx:
    print(f"{CLASS_NAMES[idx]}: {predicted_proba[idx]:.3f}")



Predicted Disease: Bronchial Asthma

Top-3 Predictions:
Bronchial Asthma: 0.801
Hepatitis C: 0.050
GERD: 0.015


# Part 3 - Suggest the home remedy

In [26]:
import pandas as pd

# ------------------------
# 1. Load home remedies CSV
# ------------------------
REMEDY_CSV_PATH = "C://Users//kalya//Desktop//Kalyan//Upgrad AIML//Thesis Topics//home_remedy.csv"   # keep relative for portability
remedy_df = pd.read_csv(REMEDY_CSV_PATH)

# ------------------------
# 2. Normalization function
# ------------------------
def normalize_text(text):
    """
    Normalize disease names for consistent matching.
    """
    return (
        text.lower()
            .strip()
            .replace("(", "")
            .replace(")", "")
            .replace("_", " ")
            .replace("  ", " ")
    )

# ------------------------
# 3. Preprocess disease column
# ------------------------
remedy_df["Disease_list"] = remedy_df["Disease"].apply(
    lambda x: [normalize_text(d) for d in x.split("/")]
)

# -----------------------------------------------------------------------------------------------------------
# 4. Remedy lookup function. Returns the home remedy if known. Otherwise, displays 'no home remedy available'
# ----------------------------------------------------------------------------------------------------------
def get_home_remedy(disease_name):
    disease_norm = normalize_text(disease_name)

    matches = remedy_df[
        remedy_df["Disease_list"].apply(lambda diseases: disease_norm in diseases)
    ]

    if not matches.empty:
        remedies = []

        for _, row in matches.iterrows():
            remedies.append(row["Home_Remedy"])

        # Remove duplicates while preserving order
        combined_remedy = " | ".join(dict.fromkeys(remedies))

        return {
            "Disease": disease_name,
            "Home_Remedy": combined_remedy,
            "Disclaimer": (
                "Home remedies are for supportive care only and do not "
                "replace professional medical treatment."
            )
        }

    return {
        "Disease": disease_name,
        "Home_Remedy": "No home remedy information available.",
        "Disclaimer": (
            "Please consult a qualified healthcare professional "
            "for medical advice."
        )
    }




In [28]:
# This step is needed to map the doctor to the specialization

DOCTOR_SPECIALTY_MAP = {

    # Infectious & Parasitic Diseases
    "fungal infection": "Dermatologist",
    "malaria": "General Physician / Infectious Disease Specialist",
    "dengue": "General Physician / Infectious Disease Specialist",
    "typhoid": "General Physician / Infectious Disease Specialist",
    "tuberculosis": "Pulmonologist",
    "chicken pox": "General Physician",
    "impetigo": "Dermatologist",
    "aids": "Infectious Disease Specialist",

    # Gastrointestinal & Liver Disorders
    "gerd": "Gastroenterologist",
    "peptic ulcer disease": "Gastroenterologist",
    "gastroenteritis": "Gastroenterologist",
    "chronic cholestasis": "Gastroenterologist",
    "jaundice": "Gastroenterologist",
    "hepatitis a": "Gastroenterologist",
    "hepatitis b": "Gastroenterologist",
    "hepatitis c": "Gastroenterologist",
    "hepatitis d": "Gastroenterologist",
    "hepatitis e": "Gastroenterologist",
    "alcoholic hepatitis": "Gastroenterologist",

    # Endocrine & Metabolic Disorders
    "diabetes": "Endocrinologist",
    "hypoglycemia": "Endocrinologist",
    "hypothyroidism": "Endocrinologist",
    "hyperthyroidism": "Endocrinologist",

    # Cardiovascular Disorders
    "hypertension": "Cardiologist",
    "heart attack": "Cardiologist",
    "varicose veins": "Vascular Surgeon / General Surgeon",

    # Respiratory Disorders
    "bronchial asthma": "Pulmonologist",
    "pneumonia": "Pulmonologist",

    # Neurological Disorders
    "migraine": "Neurologist",
    "paralysis brain hemorrhage": "Neurologist",
    "paroxysmal positional vertigo": "Neurologist / ENT Specialist",

    # Musculoskeletal Disorders
    "cervical spondylosis": "Orthopaedic Specialist",
    "osteoarthritis": "Orthopaedic Specialist",
    "arthritis": "Rheumatologist / Orthopaedic Specialist",

    # Skin Disorders
    "psoriasis": "Dermatologist",
    "acne": "Dermatologist",
    "drug reaction": "Dermatologist",

    # Urological Disorders
    "urinary tract infection": "Urologist",

    # Allergic / Immunological
    "allergy": "Allergist / Immunologist",

    # Haematological / Metabolic (supportive)
    "hypoglycemia": "Endocrinologist"
}


In [30]:
#If disease is identified in the doctor speciality mapping above, the specializaiton would be returned. Otherwise, it suggests a General Physician
def suggest_doctor(disease_name):
    disease_norm = normalize_text(disease_name)

    for key, doctor in DOCTOR_SPECIALTY_MAP.items():
        if key in disease_norm:
            return doctor

    return "General Physician"


In [32]:
#Function that generates the Conversational response to user with home remedy, caution and doctor specialization suggestion

def generate_conversational_response(disease_name):
    remedy_result = get_home_remedy(disease_name)
    doctor_type = suggest_doctor(disease_name)

    response = f"""
Based on the symptoms you shared, the condition that best matches your inputs is **{disease_name}**.

Please keep in mind that this is an AI-assisted prediction intended to support early awareness, not a medical diagnosis.

Here are some supportive home-care measures that may help you feel better:
👉 {remedy_result['Home_Remedy']}

If your symptoms persist, worsen, or cause discomfort, it would be a good idea to consult a **{doctor_type}**, who specialises in conditions like this.

{remedy_result['Disclaimer']}
"""

    return response.strip()


In [34]:
# ------------------------
# 6. Running the response generator from Main
# ------------------------

if __name__ == "__main__":

    # predicted_disease should already come from XGBoost prediction
    # Example:
    # predicted_disease = "Dengue"

    final_response = generate_conversational_response(predicted_disease)

    print("\n" + "=" * 80)
    print("AI Health Assistant")
    print("=" * 80)
    print(final_response)
    print("=" * 80)



AI Health Assistant
Based on the symptoms you shared, the condition that best matches your inputs is **Bronchial Asthma**.

Please keep in mind that this is an AI-assisted prediction intended to support early awareness, not a medical diagnosis.

Here are some supportive home-care measures that may help you feel better:
👉 Mix 1 teaspoon honey with ½ teaspoon cinnamon (dalchini) powder and have at night before going to bed. Boil carom seeds (ajwain) in water and inhale the steam. Boil 8-10 flakes of garlic (lasan) in ½ cup of milk. Have this every night. Gives excellent results in early stages of asthma. Add a handful of drumstick leaves (sahijan) to 1 cup water. Boil. Simmer on loflame for 3-4 minutes. Cool and strain. Add salt, pepper and lemon juice to taste. Have every day, once or twice a day. wAn expectorant and a very effective remedy for asthma is prepared by boiling 6 cloves (laung) in 3 tablespoons of water. Take 1 teaspoon of this decoction with a little honey, thrice daily. 